In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os
from datetime import datetime
from nba_api.stats.endpoints import leaguedashteamstats

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict

pd.set_option('display.max_columns', None)

In [3]:
today = datetime.today().strftime('%Y%m%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

us_file = get_latest_file('NBA_US_*.csv')
dfs_file = get_latest_file('NBA_DFS_*.csv')

if us_file is None:
    raise ValueError("No US file found")

if dfs_file is None:
    raise ValueError("No DFS file found")

us_df = pd.read_csv(us_file)
lines_dfs = pd.read_csv(dfs_file)

lines_us = us_df[us_df['CATEGORY'] == 'player_points'].copy()
# lines_dfs = dfs_df[
#     (dfs_df['BOOKMAKER'] == 'PrizePicks') &
#     (dfs_df['CATEGORY'] == 'player_points')
# ].copy()

print("US file:", us_file.name)
print("DFS file:", dfs_file.name)
print("DFS latest pull:", lines_dfs['DATA_PULLED_AT'].max())
print("US latest pull:", us_df['DATA_PULLED_AT'].max())

US file: NBA_US_20260319_141026.csv
DFS file: NBA_DFS_20260319_140855.csv
DFS latest pull: 2026-03-19 14:08:55
US latest pull: 2026-03-19 14:10:26


In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')

if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260319_141026.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Detroit Pistons,2026-03-19 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Charlotte Hornets,Orlando Magic,2026-03-19 23:15:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Chicago Bulls,Cleveland Cavaliers,2026-03-20 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,New Orleans Pelicans,Los Angeles Clippers,2026-03-20 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Miami Heat,Los Angeles Lakers,2026-03-20 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
df.head()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION
22480,22480,2025-26,1630611,Gui Santos,Gui,1610612744,GSW,Golden State Warriors,22500002,2025-10-21T00:00:00,GSW @ LAL,W,2.646667,0,0,0.000,0,0,0.000,0,0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,-1.0,0,0,0.0,1,2:39,1,94.9,100.0,100.0,123.0,120.0,120.0,-28.0,-20.0,-20.0,0.000,0.0,0.0,0.000,0.000,0.000,100.0,100.0,0.000,0.000,0.167,0.158,101.56,99.75,83.12,99.75,-0.111,6,0.0,0.0,38,78,0.487,17,40,0.425,26,29,0.897,9,31,40,29,19.0,10,4,2,27,21,119,10.0,118.1,119.0,106.5,106.9,11.6,12.1,0.763,1.53,21.0,0.209,0.744,0.477,0.190,0.596,0.656,101.5,101.00,84.17,100,0.546,1610612744,GSW,Golden State Warriors,38,78,0.487,17,40,0.425,26,29,0.897,9,31,40,29,19.0,10,4,2,27,21,119,10.0,118.1,119.0,106.5,106.9,11.6,12.1,0.763,1.53,21.0,0.209,0.744,0.477,0.190,0.596,0.656,101.5,101.00,84.17,100,0.546,NaN
22457,22457,2025-26,1627936,Alex Caruso,Alex,1610612760,OKC,Oklahoma City Thunder,22500001,2025-10-21T00:00:00,OKC vs. HOU,W,30.180000,3,9,0.333,2,6,0.333,0,0,0.0,0,2,2,3,0,2,1,1,2,1,8,-15,23.9,0,0,21.0,1,30:11,1,95.2,94.9,94.9,105.4,109.2,109.2,-10.2,-14.3,-14.3,0.167,0.0,25.0,0.000,0.077,0.036,0.0,0.0,0.444,0.444,0.141,0.146,100.36,98.61,82.17,98.61,0.076,59,3.0,9.0,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12.0,12,4,5,27,26,125,1.0,107.8,112.6,103.6,107.8,4.1,4.8,0.630,2.42,18.5,0.283,0.518,0.397,0.108,0.505,0.543,97.5,93.52,77.93,111,0.521,1610612760,OKC,Oklahoma City Thunder,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12.0,12,4,5,27,26,125,1.0,107.8,112.6,103.6,107.8,4.1,4.8,0.630,2.42,18.5,0.283,0.518,0.397,0.108,0.505,0.543,97.5,93.52,77.93,111,0.521,NaN
22456,22456,2025-26,1628392,Isaiah Hartenstein,Isaiah,1610612760,OKC,Oklahoma City Thunder,22500001,2025-10-21T00:00:00,OKC vs. HOU,W,36.841667,2,5,0.400,0,1,0.000,2,2,1.0,5,3,8,5,1,2,0,0,6,5,6,9,28.1,0,0,23.0,1,36:51,1,113.5,121.7,121.7,100.3,102.7,102.7,13.2,19.0,19.0,0.172,5.0,41.7,0.125,0.079,0.103,8.3,8.4,0.400,0.510,0.083,0.082,96.96,92.50,77.09,92.50,0.071,69,2.0,5.0,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12.0,12,4,5,27,26,125,1.0,107.8,112.6,103.6,107.8,4.1,4.8,0.630,2.42,18.5,0.283,0.518,0.397,0.108,0.505,0.543,97.5,93.52,77.93,111,0.521,1610612760,OKC,Oklahoma City Thunder,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12.0,12,4,5,27,26,125,1.0,107.8,112.6,103.6,107.8,4.1,4.8,0.630,2.42,18.5,0.283,0.518,0.397,0.108,0.505,0.543,97.5,93.52,77.93,111,0.521,C
22455,22455,2025-26,1627741,Buddy

In [6]:
# ── Parse game odds: consensus spread & total per team ────────────────────
game_rows = []

for game in team_dds.to_dict('records'):
    home = game['home_team']
    away = game['away_team']
    commence = game['commence_time']
    bookmakers = game['bookmakers']

    spreads_home, spreads_away, totals = [], [], []

    for bk in bookmakers:
        for market in bk['markets']:
            if market['market_key'] == 'spreads':
                for outcome in market['outcomes']:
                    if outcome['name'] == home:
                        spreads_home.append(outcome['point'])
                    elif outcome['name'] == away:
                        spreads_away.append(outcome['point'])
            elif market['market_key'] == 'totals':
                for outcome in market['outcomes']:
                    if outcome['name'] == 'Over':          # one side is enough
                        totals.append(outcome['point'])

    # Consensus = median across bookmakers; count = how many books reported
    consensus_total      = round(np.median(totals), 1)      if totals        else None
    consensus_spread_home = round(np.median(spreads_home), 1) if spreads_home else None
    consensus_spread_away = round(np.median(spreads_away), 1) if spreads_away else None
    n_books              = len(bookmakers)

    game_rows.append({
        'TEAM': home, 'OPPONENT': away,
        'TEAM_SPREAD': consensus_spread_home,
        'GAME_TOTAL':  consensus_total,
        'N_BOOKS':     n_books,
        'COMMENCE_TIME': commence,
        'HOME_AWAY': 'HOME',
    })
    game_rows.append({
        'TEAM': away, 'OPPONENT': home,
        'TEAM_SPREAD': consensus_spread_away,
        'GAME_TOTAL':  consensus_total,
        'N_BOOKS':     n_books,
        'COMMENCE_TIME': commence,
        'HOME_AWAY': 'AWAY',
    })

game_odds_df = pd.DataFrame(game_rows)

# ── Filter to players with Underdog lines ──────────────────────────────────
updated_names = []
for name in lines_dfs[(lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == 'player_points')]['NAME'].unique():
    updated_names.append(nameDict.get(name, name))

df = df[df['PLAYER_NAME'].isin(updated_names)].copy()

# ── Rolling stats (last 5 / last 10) ──────────────────────────────────────
df['AVG_MIN_L5']  = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).mean().round(2))
df['STD_MIN_L5']  = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).std().round(2))
df['AVG_PTS_L5']  = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(5).mean().round(2))
df['STD_PTS_L5']  = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(5).std().round(2))
df['MED_PTS_L5']  = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(5).median().round(2))
df['STD_PTS_L10'] = df.groupby('PLAYER_ID')['PTS'].transform(lambda x: x.rolling(10).std().round(2))

df['MIN_CONSISTENCY'] = (df['AVG_MIN_L5'] / df['STD_MIN_L5']).round(2)

# ── Most recent row per player ─────────────────────────────────────────────
latest = df.groupby('PLAYER_ID').last().reset_index()

# ── Underdog lines ─────────────────────────────────────────────────────────
prop_pts = (
    lines_dfs[(lines_dfs['BOOKMAKER'] == 'PrizePicks') & (lines_dfs['CATEGORY'] == 'player_points')]
    [['NAME', 'LINE', 'ODDS', 'COMMENCE_TIME']]
    .rename(columns={'NAME': 'PLAYER_NAME'})
    .drop_duplicates('PLAYER_NAME')
)

merged = latest.merge(prop_pts, on='PLAYER_NAME', how='inner')

# ── Real book odds (best available per side) ───────────────────────────────
real_odds = lines_us[lines_us['CATEGORY'] == 'player_points'].rename(columns={'NAME': 'PLAYER_NAME'})

for side, col in [('Over', 'ODDS_OVER'), ('Under', 'ODDS_UNDER')]:
    best = (
        real_odds[real_odds['OVER/UNDER'] == side]
        .groupby(['PLAYER_NAME', 'LINE'])['ODDS'].max()
        .reset_index().rename(columns={'ODDS': col})
    )
    merged = merged.merge(best, on=['PLAYER_NAME', 'LINE'], how='left')

merged['ODDS_OVER']  = merged['ODDS_OVER'].fillna(-137).astype(int)
merged['ODDS_UNDER'] = merged['ODDS_UNDER'].fillna(-137).astype(int)

# ── Merge game-level spread & total ───────────────────────────────────────
# Assumes df / latest has a 'TEAM' column matching full team names in game_odds.json
merged = merged.merge(
    game_odds_df[['TEAM', 'OPPONENT', 'TEAM_SPREAD', 'GAME_TOTAL', 'N_BOOKS', 'HOME_AWAY']],
    left_on='TEAM_NAME',
    right_on='TEAM',
    how='left'
).drop(columns='TEAM')

# ── Implied probability from book odds ────────────────────────────────────
def implied_prob(american_odds):
    if american_odds > 0:
        return round(100 / (american_odds + 100), 3)
    else:
        return round(abs(american_odds) / (abs(american_odds) + 100), 3)

merged['IMP_PROB_OVER']  = merged['ODDS_OVER'].apply(implied_prob)
merged['IMP_PROB_UNDER'] = merged['ODDS_UNDER'].apply(implied_prob)

# ── Core metrics ───────────────────────────────────────────────────────────
merged['EDGE']     = (merged['AVG_PTS_L5'] - merged['LINE']).round(2)
merged['MED_EDGE'] = (merged['MED_PTS_L5'] - merged['LINE']).round(2)
merged['Z_SCORE']  = ((merged['LINE'] - merged['AVG_PTS_L5']) / merged['STD_PTS_L10']).round(3)

merged['PROB_OVER']  = (1 - stats.norm.cdf(merged['Z_SCORE'])).round(3)
merged['PROB_UNDER'] = stats.norm.cdf(merged['Z_SCORE']).round(3)

# ── Game-context features ──────────────────────────────────────────────────
# Pace/scoring environment: high total → more pts available league-wide that game
merged['TOTAL_BOOST'] = ((merged['GAME_TOTAL'] - 220) / 10).round(3)   # ~0 at league avg

# Spread proxy for role/usage: big favorite → star plays less 4Q, dog → may chase
# Positive spread = underdog (gets points), negative = favorite
merged['IS_UNDERDOG'] = (merged['TEAM_SPREAD'] > 0).astype(int)

# ── Cover rate (last 10) ───────────────────────────────────────────────────
cover_df = df.merge(merged[['PLAYER_NAME', 'LINE']], on='PLAYER_NAME', how='inner')

cover_windows = {'L5': 5, 'L10': 10, 'L15': 15}

cover = cover_df.groupby('PLAYER_NAME').apply(
    lambda g: pd.Series({
        'OVER_RATE_L5':  (g['PTS'].tail(5)  > g['LINE'].iloc[0]).mean().round(2),
        'OVER_RATE_L10': (g['PTS'].tail(10) > g['LINE'].iloc[0]).mean().round(2),
        'OVER_RATE_L15': (g['PTS'].tail(15) > g['LINE'].iloc[0]).mean().round(2),
        'OVER_RATE_SEASON': (g['PTS'] > g['LINE'].iloc[0]).mean().round(2),
    })
).reset_index()

merged = merged.merge(cover, on='PLAYER_NAME', how='left')

# ── EV % ───────────────────────────────────────────────────────────────────
def calc_ev(prob, american_odds):
    decimal = (american_odds / 100 + 1) if american_odds > 0 else (100 / abs(american_odds) + 1)
    return round(((prob * (decimal - 1)) - (1 - prob)) * 100, 2)

merged['EV_OVER']  = merged.apply(lambda r: calc_ev(r['PROB_OVER'],  r['ODDS_OVER']),  axis=1)
merged['EV_UNDER'] = merged.apply(lambda r: calc_ev(r['PROB_UNDER'], r['ODDS_UNDER']), axis=1)

# ── Filters + scoring ──────────────────────────────────────────────────────
merged = merged[(merged['AVG_MIN_L5'] >= 20) & (merged['STD_MIN_L5'] <= 8)]

merged['CONFIDENCE'] = (
    (merged['EDGE'] / merged['STD_PTS_L5'])
    + merged['OVER_RATE_L10']
    + merged['MIN_CONSISTENCY'] * 0.1
    + merged['TOTAL_BOOST'] * 0.15      # slight bump in high-scoring games
).round(2)

merged['BET_FLAG'] = (
    (merged['EDGE']           >  1.5) &
    (merged['OVER_RATE_L10'] >= 0.6) &
    (merged['STD_PTS_L5']     <  6.0) &
    (merged['PROB_OVER']      >= 0.60) &
    (merged['EV_OVER']        >  0)
)

# ── Output ─────────────────────────────────────────────────────────────────
output = merged[[
    'PLAYER_NAME', 'TEAM_NAME', 'OPPONENT', 'HOME_AWAY',  # ← TEAM_NAME here
    'TEAM_SPREAD', 'GAME_TOTAL',
    'LINE', 'ODDS_OVER', 'ODDS_UNDER',
    'IMP_PROB_OVER', 'IMP_PROB_UNDER',
    'AVG_PTS_L5', 'MED_PTS_L5', 'STD_PTS_L5','EDGE', 'MED_EDGE', 'Z_SCORE',
    'PROB_OVER', 'PROB_UNDER', 'EV_OVER', 'EV_UNDER',
    'OVER_RATE_L5', 'OVER_RATE_L10', 'OVER_RATE_L15', 'OVER_RATE_SEASON', 
    'AVG_MIN_L5', 'STD_MIN_L5',
    'MIN_CONSISTENCY', 'TOTAL_BOOST', 'IS_UNDERDOG',
    'CONFIDENCE', 'BET_FLAG', 'COMMENCE_TIME'
]].sort_values('EV_OVER', ascending=False)

tier1 = output[output['BET_FLAG']]

print(f"Total players: {len(output)}")
print(f"Tier 1 bets:   {len(tier1)}\n")
output.head(2)

Total players: 48
Tier 1 bets:   2



/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_81853/1141488934.py:134: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(


,PLAYER_NAME,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_PTS_L5,MED_PTS_L5,STD_PTS_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,MIN_CONSISTENCY,TOTAL_BOOST,IS_UNDERDOG,CONFIDENCE,BET_FLAG,COMMENCE_TIME
29,Tre Jones,Chicago Bulls,Cleveland Cavaliers,HOME,11.0,238.5,12.5,-113,-104,0.531,0.510,16.8,18.0,6.38,4.3,5.5,-0.862,0.806,0.194,51.93,-61.95,0.8,0.8,0.53,0.44,29.43,3.81,7.72,1.85,1,2.52,False,2026-03-20
48,Matas Buzelis,Chicago Bulls,Cleveland Cavaliers,HOME,11.0,238.5,19.5,-104,-108,0.510,0.519,25.8,22.0,9.52,6.3,2.5,-0.739,0.770,0.230,51.04,-55.70,0.6,0.7,0.47,0.32,35.53,5.71,6.22,1.85,1,2.26,False,2026-03-20


In [7]:
league_df  = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]
team_stats = league_df.set_index('TEAM_ID')

opp_stats = (
    league_df[['TEAM_NAME', 'DEF_RATING', 'DEF_RATING_RANK', 'PACE', 'PACE_RANK']]
    .copy()
    .rename(columns={
        'TEAM_NAME':  'OPPONENT',      
        'DEF_RATING': 'OPP_DEF_RATING',
        'DEF_RATING_RANK': 'OPP_RANK_DEF_RATING',
        'PACE': 'OPP_PACE',
        'PACE_RANK' : 'OPP_PACE_RANK'}))

final = output.merge(opp_stats, on='OPPONENT', how='left')
final = final[[
    'PLAYER_NAME', 'LINE', 'OPPONENT',
    'TEAM_SPREAD', 'GAME_TOTAL', 'OPP_DEF_RATING',	'OPP_RANK_DEF_RATING',	'OPP_PACE',	'OPP_PACE_RANK',
    'ODDS_OVER', 'ODDS_UNDER',
    'IMP_PROB_OVER', 'IMP_PROB_UNDER',
    'AVG_PTS_L5', 'MED_PTS_L5', 'STD_PTS_L5','EDGE', 'MED_EDGE', 'Z_SCORE',
    'PROB_OVER', 'PROB_UNDER', 'EV_OVER', 'EV_UNDER',
    'OVER_RATE_L5', 'OVER_RATE_L10', 'OVER_RATE_L15', 'OVER_RATE_SEASON', 
    'AVG_MIN_L5', 'STD_MIN_L5','MIN_CONSISTENCY', 'IS_UNDERDOG']].sort_values('EV_OVER', ascending=False)
final.head()

,PLAYER_NAME,LINE,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_PTS_L5,MED_PTS_L5,STD_PTS_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,MIN_CONSISTENCY,IS_UNDERDOG
0,Tre Jones,12.5,Cleveland Cavaliers,11.0,238.5,113.4,13.0,100.73,13.0,-113,-104,0.531,0.510,16.8,18.0,6.38,4.3,5.5,-0.862,0.806,0.194,51.93,-61.95,0.8,0.8,0.53,0.44,29.43,3.81,7.72,1
1,Matas Buzelis,19.5,Cleveland Cavaliers,11.0,238.5,113.4,13.0,100.73,13.0,-104,-108,0.510,0.519,25.8,22.0,9.52,6.3,2.5,-0.739,0.770,0.230,51.04,-55.70,0.6,0.7,0.47,0.32,35.53,5.71,6.22,1
2,Keon Ellis,8.5,Chicago Bulls,-11.0,238.5,116.8,23.0,102.53,4.0,100,-110,0.500,0.524,13.0,13.0,6.60,4.5,4.5,-0.665,0.747,0.253,49.40,-51.70,0.6,0.4,0.33,0.27,27.66,3.36,8.23,0
3,Bam Adebayo,20.5,Los Angeles Lakers,-5.5,240.5,115.8,20.0,99.32,21.0,-106,-107,0.515,0.517,34.4,24.0,27.23,13.9,3.5,-0.720,0.764,0.236,48.48,-54.34,0.8,0.8,0.73,0.45,37.59,3.15,11.93,0
4,Devin Booker,27.5,San Antonio Spurs,9.5,228.0,110.4,3.0,100.80,12.0,-110,-109,0.524,0.522,35.0,34.0,6.52,7.5,6.5,-0.684,0.753,0.247,43.75,-52.64,0.8,0.6,0.47,0.38,34.92,1.47,23.76,1


In [8]:
rename_map = {
    'PLAYER_NAME': 'Player',
    'LINE': 'Line',
    'OPPONENT': 'Opponent',
    'TEAM_SPREAD': 'Spread',
    'GAME_TOTAL': 'Total',
    'OPP_DEF_RATING': 'Opp Def Rating',
    'OPP_RANK_DEF_RATING': 'Opp Def Rank',
    'OPP_PACE': 'Opp Pace',
    'OPP_PACE_RANK': 'Opp Pace Rank',
    'ODDS_OVER': 'Odds Over',
    'ODDS_UNDER': 'Odds Under',
    'IMP_PROB_OVER': 'Implied Over',
    'IMP_PROB_UNDER': 'Implied Under',
    'AVG_PTS_L5': 'Avg Pts L5',
    'MED_PTS_L5': 'Med Pts L5',
    'STD_PTS_L5': 'Std Pts L5',
    'EDGE': 'Edge',
    'MED_EDGE': 'Med Edge',
    'Z_SCORE': 'Z Score',
    'PROB_OVER': 'Prob Over',
    'PROB_UNDER': 'Prob Under',
    'EV_OVER': 'EV Over',
    'EV_UNDER': 'EV Under',
    'OVER_RATE_L5': 'OVER L5',
    'OVER_RATE_L10': 'OVER L10',
    'OVER_RATE_L15': 'OVER L15',
    'OVER_RATE_SEASON': 'ALL SEASON',
    'AVG_MIN_L5': 'Avg Min L5',
    'STD_MIN_L5': 'Std Min L5',
    'MIN_CONSISTENCY': 'Min Consistency',
    'IS_UNDERDOG': 'Underdog'
}

df = final.rename(columns=rename_map)
df.head()

,Player,Line,Opponent,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank,Odds Over,Odds Under,Implied Over,Implied Under,Avg Pts L5,Med Pts L5,Std Pts L5,Edge,Med Edge,Z Score,Prob Over,Prob Under,EV Over,EV Under,OVER L5,OVER L10,OVER L15,ALL SEASON,Avg Min L5,Std Min L5,Min Consistency,Underdog
0,Tre Jones,12.5,Cleveland Cavaliers,11.0,238.5,113.4,13.0,100.73,13.0,-113,-104,0.531,0.510,16.8,18.0,6.38,4.3,5.5,-0.862,0.806,0.194,51.93,-61.95,0.8,0.8,0.53,0.44,29.43,3.81,7.72,1
1,Matas Buzelis,19.5,Cleveland Cavaliers,11.0,238.5,113.4,13.0,100.73,13.0,-104,-108,0.510,0.519,25.8,22.0,9.52,6.3,2.5,-0.739,0.770,0.230,51.04,-55.70,0.6,0.7,0.47,0.32,35.53,5.71,6.22,1
2,Keon Ellis,8.5,Chicago Bulls,-11.0,238.5,116.8,23.0,102.53,4.0,100,-110,0.500,0.524,13.0,13.0,6.60,4.5,4.5,-0.665,0.747,0.253,49.40,-51.70,0.6,0.4,0.33,0.27,27.66,3.36,8.23,0
3,Bam Adebayo,20.5,Los Angeles Lakers,-5.5,240.5,115.8,20.0,99.32,21.0,-106,-107,0.515,0.517,34.4,24.0,27.23,13.9,3.5,-0.720,0.764,0.236,48.48,-54.34,0.8,0.8,0.73,0.45,37.59,3.15,11.93,0
4,Devin Booker,27.5,San Antonio Spurs,9.5,228.0,110.4,3.0,100.80,12.0,-110,-109,0.524,0.522,35.0,34.0,6.52,7.5,6.5,-0.684,0.753,0.247,43.75,-52.64,0.8,0.6,0.47,0.38,34.92,1.47,23.76,1


In [12]:
df = df[[
'Player',
'Opponent',
'Line',
'Odds Over',
'Odds Under',
'Implied Over',
'Implied Under',
'EV Over',
'EV Under',
'Avg Pts L5',
'Std Pts L5',
'Prob Over',
'Prob Under',
'OVER L5',
'OVER L10',
'OVER L15',
'Avg Min L5',
'Spread',
'Total',
'Opp Def Rating',
'Opp Def Rank',
'Opp Pace',
'Opp Pace Rank'
]].sort_values(by='EV Over', ascending=False)
df.head()

,Player,Opponent,Line,Odds Over,Odds Under,Implied Over,Implied Under,EV Over,EV Under,Avg Pts L5,Std Pts L5,Prob Over,Prob Under,OVER L5,OVER L10,OVER L15,Avg Min L5,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank
0,Tre Jones,Cleveland Cavaliers,12.5,-113,-104,0.531,0.510,51.93,-61.95,16.8,6.38,0.806,0.194,0.8,0.8,0.53,29.43,11.0,238.5,113.4,13.0,100.73,13.0
1,Matas Buzelis,Cleveland Cavaliers,19.5,-104,-108,0.510,0.519,51.04,-55.70,25.8,9.52,0.770,0.230,0.6,0.7,0.47,35.53,11.0,238.5,113.4,13.0,100.73,13.0
2,Keon Ellis,Chicago Bulls,8.5,100,-110,0.500,0.524,49.40,-51.70,13.0,6.60,0.747,0.253,0.6,0.4,0.33,27.66,-11.0,238.5,116.8,23.0,102.53,4.0
3,Bam Adebayo,Los Angeles Lakers,20.5,-106,-107,0.515,0.517,48.48,-54.34,34.4,27.23,0.764,0.236,0.8,0.8,0.73,37.59,-5.5,240.5,115.8,20.0,99.32,21.0
4,Devin Booker,San Antonio Spurs,27.5,-110,-109,0.524,0.522,43.75,-52.64,35.0,6.52,0.753,0.247,0.8,0.6,0.47,34.92,9.5,228.0,110.4,3.0,100.80,12.0


In [10]:
output_path = f'data/props/ev_analysis/prizepicks.csv'
df.to_csv(output_path, index=False)